In [ ]:
VOLUME_PATH = "/Volumes/workspace/default/kthdv&dtdm"

In [ ]:
# Hàm đọc
def read_csv(file_name):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{VOLUME_PATH}/{file_name}")
    )

# Load dữ liệu
customers_df = read_csv("olist_customers_dataset.csv")

orders_df = read_csv("olist_orders_dataset.csv")

products_df = read_csv("olist_products_dataset.csv")

sellers_df = read_csv("olist_sellers_dataset.csv")

reviews_df = read_csv("olist_order_reviews_dataset.csv")

payments_df = read_csv("olist_order_payments_dataset.csv")

items_df = read_csv("olist_order_items_dataset.csv")

category_df = read_csv("product_category_name_translation.csv")

# Keep immutable raw DataFrames for the Bronze layer.
raw_products_df = products_df
raw_reviews_df = reviews_df

In [ ]:
reviews_df.count()
products_df.count()
orders_df.count()

Data Cleaning

In [ ]:
# Clean reviews and keep one deterministic review per order.
from pyspark.sql import Window
from pyspark.sql.functions import col, expr, row_number, to_timestamp

reviews_with_score_df = reviews_df.withColumn(
    "_review_score",
    expr("try_cast(review_score AS INT)")
)

invalid_review_scores = reviews_with_score_df.filter(
    col("review_score").isNotNull() &
    (col("_review_score").isNull() | (~col("_review_score").between(1, 5)))
).count()

assert invalid_review_scores == 0, (
    "Invalid review_score values found. Read the review CSV with multiLine=True."
)

reviews_df = (
    reviews_with_score_df
    .filter(col("order_id").isNotNull())
    .drop("review_score")
    .withColumnRenamed("_review_score", "review_score")
    .filter(col("review_score").between(1, 5))
    .dropDuplicates()
)

review_order_window = Window.partitionBy("order_id").orderBy(
    col("_review_answer_at").desc_nulls_last(),
    col("_review_creation_at").desc_nulls_last(),
    col("review_id").desc_nulls_last()
)

reviews_order_df = (
    reviews_df
    .withColumn(
        "_review_answer_at",
        to_timestamp("review_answer_timestamp")
    )
    .withColumn(
        "_review_creation_at",
        to_timestamp("review_creation_date")
    )
    .withColumn("_row_number", row_number().over(review_order_window))
    .filter(col("_row_number") == 1)
    .drop("_row_number", "_review_answer_at", "_review_creation_at")
)


In [ ]:
# Clean Products
# Điền category thiếu
from pyspark.sql.functions import lit

products_df = products_df.fillna({
    "product_category_name": "unknown"
})

# Điền numeric missing
products_df = products_df.fillna({
    "product_name_lenght": 0,
    "product_description_lenght": 0,
    "product_photos_qty": 0,
    "product_weight_g": 0,
    "product_length_cm": 0,
    "product_height_cm": 0,
    "product_width_cm": 0
})

In [ ]:
# Clean Orders
# Không xóa các bản ghi có:
# order_delivered_customer_date
# order_delivered_carrier_date
# Đó là đơn bị:canceled, unavailable, processing
# và chứa thông tin hành vi quan trọng.

In [ ]:
# Save Bronze Layer
customers_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/customers"
)

orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/orders"
)

raw_products_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/products"
)

sellers_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/sellers"
)

raw_reviews_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/reviews"
)

payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/payments"
)

items_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/order_items"
)

category_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/bronze/category_translation"
)

Data Transformation

In [ ]:
from pyspark.sql.functions import *
# Orders + Customers
# Customer Order Dataset
orders_customers_df = (
    orders_df.alias("o")
    .join(
        customers_df.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    )
    .select(
        "o.*",
        "c.customer_unique_id",
        "c.customer_city",
        "c.customer_state"
    )
)

display(orders_customers_df.limit(5))

In [ ]:
# Orders + Reviews
# Review Dataset
orders_reviews_df = (
    orders_customers_df.alias("o")
    .join(
        reviews_order_df.alias("r"),
        "order_id",
        "left"
    )
)

display(orders_reviews_df.limit(5))

In [ ]:
# Aggregate Payments
# Một order có thể có nhiều payment record nên phải aggregate trước khi join.
payments_agg_df = (
    payments_df
    .groupBy("order_id")
    .agg(
        sum("payment_value").alias("total_payment_value"),
        first("payment_type").alias("payment_type"),
        max("payment_installments").alias("payment_installments")
    )
)

display(payments_agg_df.limit(5))

In [ ]:
# Join Payment
orders_payment_df = (
    orders_reviews_df.alias("o")
    .join(
        payments_agg_df.alias("p"),
        "order_id",
        "left"
    )
)

display(orders_payment_df.limit(5))

In [ ]:
# Product Dataset
# order_items + products + sellers + category_translation
product_master_df = (
    items_df.alias("oi")
    .join(
        products_df.alias("p"),
        "product_id",
        "left"
    )
    .join(
        sellers_df.alias("s"),
        "seller_id",
        "left"
    )
    .join(
        category_df.alias("ct"),
        "product_category_name",
        "left"
    )
)

In [ ]:
product_master_df = product_master_df.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "seller_city",
    "seller_state"
)

display(product_master_df.limit(5))

In [ ]:
# Aggregate Product Information theo Order
# Một order có thể có nhiều item.
# ML sẽ dự đoán theo đơn hàng nên cần gom lại.

order_product_agg_df = (
    product_master_df
    .groupBy("order_id")
    .agg(
        count("*").alias("number_of_items"),
        sum("price").alias("total_product_value"),
        sum("freight_value").alias("total_freight_value"),
        avg("price").alias("avg_item_price")
    )
)

display(order_product_agg_df.limit(5))

In [ ]:
# Tạo Master Dataset
# Đây là dataset trung tâm của dự án.

master_orders_df = (
    orders_payment_df.alias("o")
    .join(
        order_product_agg_df.alias("p"),
        "order_id",
        "left"
    )
)

In [ ]:
# Validate the order-level grain before saving Silver tables.
orders_count = orders_df.select("order_id").distinct().count()
master_rows = master_orders_df.count()
master_unique_orders = master_orders_df.select("order_id").distinct().count()
duplicate_master_orders = master_rows - master_unique_orders

print("Orders:", orders_count)
print("Master rows:", master_rows)
print("Unique master orders:", master_unique_orders)
print("Duplicated master orders:", duplicate_master_orders)
print("Columns:", len(master_orders_df.columns))

assert duplicate_master_orders == 0, "master_orders must contain one row per order_id"
assert master_unique_orders == orders_count, "master_orders must preserve every order"

display(master_orders_df.limit(20))

In [ ]:
orders_customers_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/silver/orders_customers"
)

payments_agg_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/silver/payments_agg"
)

reviews_order_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/silver/reviews_order"
)

product_master_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/silver/product_master"
)

order_product_agg_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/silver/order_product_agg"
)

master_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    "/Volumes/workspace/default/kthdv&dtdm/silver/master_orders"
)